# Homework 7: Time-series features and JSONB event-log

Data sources: NYC TLC Yellow Taxi, March 2024 (time-series part), and a synthetic event-log with events dated 2024-02-01 to 2024-03-31 (JSONB part).

## 1. Setup and loading

In [ ]:
# Setup the environment
%pip install -q pgserver psycopg2-binary sqlalchemy pandas pyarrow requests

from urllib.request import urlretrieve
import pandas as pd
import pgserver
from sqlalchemy import create_engine, text

pd.set_option('display.max_colwidth', None)

pg = pgserver.get_server('/tmp/hw7_pg', cleanup_mode='stop')

engine = create_engine(
    pg.get_uri(),
    future=True,
    connect_args={'options': '-c timezone=UTC'},
)


def explain(sql_text):
    """Run an EXPLAIN query and print the complete plan."""
    with engine.connect() as conn:
        print('\n'.join(row[0] for row in conn.execute(text(sql_text))))


with engine.connect() as conn:
    version = conn.execute(text('SELECT version()')).scalar_one()
    print(version)

Note: you may need to restart the kernel to use updated packages.
PostgreSQL 16.2 on aarch64-apple-darwin23.5.0, compiled by Apple clang version 15.0.0 (clang-1500.0.40.1), 64-bit


In [ ]:
import tempfile
from pathlib import Path

URL = (
    'https://d37ci6vzurychx.cloudfront.net/trip-data/'
    'yellow_tripdata_2024-03.parquet'
)

parquet_path = Path(tempfile.gettempdir()) / 'yellow_tripdata_2024-03.parquet'

if not parquet_path.exists():
    urlretrieve(URL, parquet_path)

columns = [
    'tpep_pickup_datetime',
    'tpep_dropoff_datetime',
    'PULocationID',
    'DOLocationID',
    'trip_distance',
    'fare_amount',
    'passenger_count',
]

df = pd.read_parquet(parquet_path, columns=columns)

# TLC files contain stray timestamps outside the month: keep only 2024-03
df = df[
    (df['tpep_pickup_datetime'] >= '2024-03-01')
    & (df['tpep_pickup_datetime'] < '2024-04-01')
]

duration = df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']
df = df[(duration > pd.Timedelta(0)) & (duration < pd.Timedelta(hours=24))]

df = df.sample(n=min(500_000, len(df)), random_state=42)

# Basic cleaning before loading to PostgreSQL
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 500)]
df = df[(df['trip_distance'] > 0) & (df['trip_distance'] < 100)]
df = df[df['PULocationID'].notna()]

df.to_sql(
    'nyc_taxi_raw',
    engine,
    if_exists='replace',
    index=False,
    chunksize=10_000,
)

print("Source: NYC Taxi TLC Parquet (2024-03), random sample")
print("Trips loaded (pandas):", len(df))
pd.read_sql('SELECT COUNT(*) AS n_rows FROM nyc_taxi_raw', engine)

Source: NYC Taxi TLC Parquet (2024-03), random sample
Trips loaded (pandas): 480300


,n_rows
0,480300


In [204]:
sql = '''
DROP TABLE IF EXISTS t7_hw_taxi_daily CASCADE;

CREATE TABLE t7_hw_taxi_daily AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE AS ts_day,
    COUNT(*)::INTEGER AS trip_count,
    AVG(fare_amount)::NUMERIC(10, 2) AS avg_fare,
    SUM(trip_distance)::NUMERIC(12, 2) AS total_distance
FROM nyc_taxi_raw
WHERE fare_amount > 0
    AND fare_amount < 500
    AND trip_distance > 0
    AND trip_distance < 100
    AND "PULocationID" IS NOT NULL
GROUP BY
    "PULocationID"::INTEGER,
    DATE_TRUNC('day', tpep_pickup_datetime::TIMESTAMP)::DATE;

ALTER TABLE t7_hw_taxi_daily
    ADD PRIMARY KEY (pickup_zone_id, ts_day);
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Daily aggregation table created successfully!")


Daily aggregation table created successfully!


In [205]:
sql = """
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day,
       MAX(ts_day) AS last_day,
       COUNT(DISTINCT ts_day) AS n_distinct_days
FROM t7_hw_taxi_daily;
"""

pd.read_sql(sql, engine)

,n_rows,n_zones,first_day,last_day,n_distinct_days
0,5438,245,2024-03-01,2024-03-31,31


In [206]:
sql = """
SELECT pickup_zone_id, COUNT(*) AS n_days
FROM t7_hw_taxi_daily
GROUP BY pickup_zone_id
ORDER BY pickup_zone_id
LIMIT 10;
"""

pd.read_sql(sql, engine)

,pickup_zone_id,n_days
0,1,12
1,2,1
2,3,15
3,4,31
4,6,1
5,7,31
6,9,9
7,10,31
8,11,17
9,12,31


In [207]:
sql = """
DROP TABLE IF EXISTS t7_hw_taxi_trips_typed;

CREATE TABLE t7_hw_taxi_trips_typed AS
SELECT
    "PULocationID"::INTEGER AS pickup_zone_id,
    "DOLocationID"::INTEGER AS dropoff_zone_id,
    (tpep_pickup_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS pickup_at,
    (tpep_dropoff_datetime::TIMESTAMP AT TIME ZONE 'America/New_York') AS dropoff_at,
    trip_distance::NUMERIC(10, 2) AS trip_distance,
    fare_amount::NUMERIC(10, 2) AS fare_amount,
    passenger_count::INTEGER AS passenger_count
FROM nyc_taxi_raw
WHERE fare_amount > 0
  AND fare_amount < 500
  AND trip_distance > 0
  AND trip_distance < 100;
"""

with engine.begin() as conn:
    conn.execute(text(sql))

print("Typed table created successfully!")

Typed table created successfully!


In [208]:
sql = """
SELECT pickup_at, pg_typeof(pickup_at) AS pickup_type
FROM t7_hw_taxi_trips_typed
LIMIT 5;
"""

pd.read_sql(sql, engine)

,pickup_at,pickup_type
0,2024-03-07 21:28:08+00:00,timestamp with time zone
1,2024-03-07 02:11:30+00:00,timestamp with time zone
2,2024-03-12 20:38:45+00:00,timestamp with time zone
3,2024-03-14 21:40:51+00:00,timestamp with time zone
4,2024-03-07 19:23:34+00:00,timestamp with time zone


The `TIMESTAMPTZ` value is an absolute instant (printed in UTC because the session time zone is UTC). Converting it back with `AT TIME ZONE 'America/New_York'` returns the original NYC wall-clock time from the TLC file.

In [209]:
sql = """
SELECT pickup_at AS pickup_at_utc,
       pickup_at AT TIME ZONE 'America/New_York' AS pickup_nyc_wall_clock
FROM t7_hw_taxi_trips_typed
LIMIT 5;
"""

pd.read_sql(sql, engine)

,pickup_at_utc,pickup_nyc_wall_clock
0,2024-03-07 21:28:08+00:00,2024-03-07 16:28:08
1,2024-03-07 02:11:30+00:00,2024-03-06 21:11:30
2,2024-03-12 20:38:45+00:00,2024-03-12 16:38:45
3,2024-03-14 21:40:51+00:00,2024-03-14 17:40:51
4,2024-03-07 19:23:34+00:00,2024-03-07 14:23:34


## 2. Time-series ML features

**Feature 1. Lag features: lag-1d, lag-7d, lag-30d**

In [210]:
sql = '''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)  OVER w AS trips_lag_1d,
    LAG(trip_count, 7)  OVER w AS trips_lag_7d,
    LAG(trip_count, 30) OVER w AS trips_lag_30d
FROM t7_hw_taxi_daily
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
'''

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,trips_lag_7d,trips_lag_30d
0,1,2024-03-03,2,NaN,NaN,None
1,1,2024-03-05,1,2.0,NaN,None
2,1,2024-03-08,1,1.0,NaN,None
3,1,2024-03-10,1,1.0,NaN,None
4,1,2024-03-15,2,1.0,NaN,None
5,1,2024-03-16,1,2.0,NaN,None
6,1,2024-03-20,1,1.0,NaN,None
7,1,2024-03-21,1,1.0,2.0,None
8,1,2024-03-22,1,1.0,1.0,None
9,1,2024-03-25,1,1.0,1.0,None


**Caveat: `LAG(x, n)` is row-based, not calendar-based.**

`t7_hw_taxi_daily` has no row for a zone/day without trips, so on sparse zones `LAG(trip_count, 1)` is the *previous observed day*, not yesterday, and `LAG(trip_count, 7)` is seven *rows* back, not seven days. The query below (zone 1) shows the real distance in days (`days_back_lag_*`): values larger than 1 and 7 mean the feature is mislabeled for that row. For dense zones (all 31 days present) the lags are exact. A production feature table would first densify the calendar (`generate_series` cross join zones, filling `trip_count = 0`), or use a range/self-join on `ts_day - INTERVAL '7 days'`. Here the limitation is kept visible instead of being hidden.

In [211]:
sql = '''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    LAG(trip_count, 1)             OVER w AS trips_lag_1d,
    ts_day - LAG(ts_day, 1)        OVER w AS days_back_lag_1d,
    LAG(trip_count, 7)             OVER w AS trips_lag_7d,
    ts_day - LAG(ts_day, 7)        OVER w AS days_back_lag_7d
FROM t7_hw_taxi_daily
WHERE pickup_zone_id = 1
WINDOW w AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
)
ORDER BY ts_day;
'''

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,trip_count,trips_lag_1d,days_back_lag_1d,trips_lag_7d,days_back_lag_7d
0,1,2024-03-03,2,NaN,NaN,NaN,NaN
1,1,2024-03-05,1,2.0,2.0,NaN,NaN
2,1,2024-03-08,1,1.0,3.0,NaN,NaN
3,1,2024-03-10,1,1.0,2.0,NaN,NaN
4,1,2024-03-15,2,1.0,5.0,NaN,NaN
5,1,2024-03-16,1,2.0,1.0,NaN,NaN
6,1,2024-03-20,1,1.0,4.0,NaN,NaN
7,1,2024-03-21,1,1.0,1.0,2.0,18.0
8,1,2024-03-22,1,1.0,1.0,1.0,17.0
9,1,2024-03-25,1,1.0,3.0,1.0,17.0


**Feature 2. Rolling 7-day average fare**

In [212]:
sql = '''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
    COUNT(*)      OVER w7 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w7 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
'''

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,avg_fare,rolling_avg_fare_7d,days_in_window
0,1,2024-03-03,51.00,51.000000,1
1,1,2024-03-05,120.00,85.500000,2
2,1,2024-03-08,105.00,92.000000,3
3,1,2024-03-10,25.00,83.333333,3
4,1,2024-03-15,18.05,21.525000,2
5,1,2024-03-16,100.00,47.683333,3
6,1,2024-03-20,105.00,74.350000,3
7,1,2024-03-21,3.00,56.512500,4
8,1,2024-03-22,115.00,80.750000,4
9,1,2024-03-25,8.60,57.900000,4


**Feature 3. Rolling 30-day fare volatility**

In [213]:
sql = '''
SELECT
    pickup_zone_id,
    ts_day,
    avg_fare,
    STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,
    COUNT(*)              OVER w30 AS days_in_window
FROM t7_hw_taxi_daily
WINDOW w30 AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day::TIMESTAMP
    RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
'''

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,avg_fare,rolling_std_fare_30d,days_in_window
0,1,2024-03-03,51.00,NaN,1
1,1,2024-03-05,120.00,48.790368,2
2,1,2024-03-08,105.00,36.290495,3
3,1,2024-03-10,25.00,44.724155,4
4,1,2024-03-15,18.05,46.417190,5
5,1,2024-03-16,100.00,44.067340,6
6,1,2024-03-20,105.00,42.365815,7
7,1,2024-03-21,3.00,46.733423,8
8,1,2024-03-22,115.00,46.680656,9
9,1,2024-03-25,8.60,48.276081,10


**Feature 4. Expanding cumulative trips**

In [214]:
sql = '''
SELECT
    pickup_zone_id,
    ts_day,
    trip_count,
    SUM(trip_count) OVER we AS cumulative_trips,
    AVG(trip_count) OVER we AS expanding_avg_trips
FROM t7_hw_taxi_daily
WINDOW we AS (
    PARTITION BY pickup_zone_id
    ORDER BY ts_day
    ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
)
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
'''

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,trip_count,cumulative_trips,expanding_avg_trips
0,1,2024-03-03,2,2,2.000000
1,1,2024-03-05,1,3,1.500000
2,1,2024-03-08,1,4,1.333333
3,1,2024-03-10,1,5,1.250000
4,1,2024-03-15,2,7,1.400000
5,1,2024-03-16,1,8,1.333333
6,1,2024-03-20,1,9,1.285714
7,1,2024-03-21,1,10,1.250000
8,1,2024-03-22,1,11,1.222222
9,1,2024-03-25,1,12,1.200000


**Combined feature view**

In [215]:
sql = '''
DROP VIEW IF EXISTS v_t7_hw_taxi_features;

CREATE VIEW v_t7_hw_taxi_features AS
WITH features_basic AS (
    SELECT
        pickup_zone_id,
        ts_day,
        trip_count,
        avg_fare,
        total_distance,

        LAG(trip_count, 1)  OVER w_pid AS trips_lag_1d,
        LAG(trip_count, 7)  OVER w_pid AS trips_lag_7d,
        LAG(trip_count, 30) OVER w_pid AS trips_lag_30d,

        AVG(avg_fare) OVER w7 AS rolling_avg_fare_7d,
        STDDEV_SAMP(avg_fare) OVER w30 AS rolling_std_fare_30d,

        SUM(trip_count) OVER we AS cumulative_trips,
        AVG(trip_count) OVER we AS expanding_avg_trips
    FROM t7_hw_taxi_daily
    WINDOW
        w_pid AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
        ),
        w7 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '6 days' PRECEDING AND CURRENT ROW
        ),
        w30 AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day::TIMESTAMP
            RANGE BETWEEN INTERVAL '29 days' PRECEDING AND CURRENT ROW
        ),
        we AS (
            PARTITION BY pickup_zone_id
            ORDER BY ts_day
            ROWS BETWEEN UNBOUNDED PRECEDING AND CURRENT ROW
        )
)
SELECT *
FROM features_basic
WHERE trips_lag_30d IS NOT NULL;
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Combined feature view created successfully!")

Combined feature view created successfully!


In [216]:
sql = """
SELECT *
FROM v_t7_hw_taxi_features
ORDER BY pickup_zone_id, ts_day
LIMIT 20;
"""

pd.read_sql(sql, engine)

,pickup_zone_id,ts_day,trip_count,avg_fare,total_distance,trips_lag_1d,trips_lag_7d,trips_lag_30d,rolling_avg_fare_7d,rolling_std_fare_30d,cumulative_trips,expanding_avg_trips
0,4,2024-03-31,35,16.17,102.61,69,51,21,16.827143,3.119564,994,32.064516
1,7,2024-03-31,15,27.85,96.90,15,13,7,30.255714,9.658305,282,9.096774
2,10,2024-03-31,6,60.10,72.00,5,6,8,54.874286,8.286447,182,5.870968
3,12,2024-03-31,8,24.09,37.98,11,7,2,27.024286,6.917042,149,4.806452
4,13,2024-03-31,40,21.08,168.29,71,61,84,21.151429,2.208027,2804,90.451613
5,17,2024-03-31,3,13.97,7.23,2,5,1,25.742857,10.981152,116,3.741935
6,24,2024-03-31,45,16.07,134.13,47,31,49,16.635714,1.744850,1262,40.709677
7,25,2024-03-31,9,30.41,77.01,6,9,10,29.500000,7.026034,251,8.096774
8,33,2024-03-31,12,29.83,79.25,8,9,12,27.250000,4.884979,321,10.354839
9,39,2024-03-31,5,28.70,34.90,4,4,8,29.797143,10.197925,145,4.677419


**Size of the feature view.** The view keeps the task's filter `trips_lag_30d IS NOT NULL`. Because the source is a single month (31 days), a row needs 30 earlier rows in its zone, so only the 31st day (2024-03-31) of zones that have all 31 days survives: one row per zone. That is enough to prove the logic, but it is a very small training set. A second month of data would give each zone ~30 additional usable days.

In [217]:
sql = """
SELECT COUNT(*) AS n_rows,
       COUNT(DISTINCT pickup_zone_id) AS n_zones,
       MIN(ts_day) AS first_day,
       MAX(ts_day) AS last_day
FROM v_t7_hw_taxi_features;
"""

pd.read_sql(sql, engine)

,n_rows,n_zones,first_day,last_day
0,84,84,2024-03-31,2024-03-31


## 3. JSONB-table for events

In [218]:
sql = '''
DROP TABLE IF EXISTS t7_hw_events;
CREATE TABLE t7_hw_events (
    event_id    INTEGER GENERATED BY DEFAULT AS IDENTITY PRIMARY KEY,
    user_id     INTEGER NOT NULL,
    event_ts    TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    event_type  TEXT NOT NULL,
    payload     JSONB NOT NULL
);

INSERT INTO t7_hw_events (user_id, event_ts, event_type, payload)
SELECT
    (n % 1000) + 1 AS user_id,
    TIMESTAMPTZ '2024-04-01 00:00:00+00'
        - ((n % 60) || ' days')::INTERVAL AS event_ts,
    CASE (n % 5)
        WHEN 0 THEN 'page_view'
        WHEN 1 THEN 'click'
        WHEN 2 THEN 'add_to_cart'
        WHEN 3 THEN 'checkout'
        ELSE        'recommendation_shown'
    END AS event_type,
    CASE (n % 5)
        WHEN 0 THEN jsonb_build_object(
            'url', '/product/' || ((n % 200) + 1),
            'referrer', (ARRAY['google','direct','email','fb'])[(n % 4) + 1],
            'device', (ARRAY['mobile','desktop','tablet'])[(n % 3) + 1]
        )
        WHEN 1 THEN jsonb_build_object(
            'element_id', 'btn_' || ((n % 50) + 1),
            'page_url', '/category/' || ((n % 20) + 1),
            'device', (ARRAY['mobile','desktop'])[(n % 2) + 1]
        )
        WHEN 2 THEN jsonb_build_object(
            'product_id', (n % 200) + 1,
            'price', ((n % 100) + 10)::NUMERIC(10, 2),
            'quantity', (n % 5) + 1,
            'tags', jsonb_build_array(
                (ARRAY['promo','new','clearance'])[(n % 3) + 1]
            )
        )
        WHEN 3 THEN jsonb_build_object(
            'order_id', 'o_' || (n + 100000),
            'total', ((n % 500) + 50)::NUMERIC(10, 2),
            'items_count', (n % 10) + 1,
            'payment_method', (ARRAY['card','paypal','crypto'])[(n % 3) + 1]
        )
        ELSE jsonb_build_object(
            'model_name', 'rec_v' || ((n % 5) + 1),
            'model_version', '1.' || (n % 10),
            'recommendations', jsonb_build_array(
                (n % 200) + 1,
                ((n + 1) % 200) + 1,
                ((n + 2) % 200) + 1
            ),
            'experiment', 'exp_' || ((n % 3) + 1)
        )
    END AS payload
FROM generate_series(1, 50000) AS n;

ANALYZE t7_hw_events;

SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Synthetic table created successfully!")

Synthetic table created successfully!


In [219]:
sql = """
SELECT event_type, COUNT(*) AS n_events
FROM t7_hw_events
GROUP BY event_type
ORDER BY event_type;
"""

pd.read_sql(sql, engine)

,event_type,n_events
0,add_to_cart,10000
1,checkout,10000
2,click,10000
3,page_view,10000
4,recommendation_shown,10000


In [220]:
sql = """
SELECT * FROM t7_hw_events LIMIT 10
"""

pd.read_sql(sql, engine)
     

,event_id,user_id,event_ts,event_type,payload
0,1,2,2024-03-31 00:00:00+00:00,click,"{'device': 'desktop', 'page_url': '/category/2', 'element_id': 'btn_2'}"
1,2,3,2024-03-30 00:00:00+00:00,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
2,3,4,2024-03-29 00:00:00+00:00,checkout,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
3,4,5,2024-03-28 00:00:00+00:00,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
4,5,6,2024-03-27 00:00:00+00:00,page_view,"{'url': '/product/6', 'device': 'tablet', 'referrer': 'direct'}"
5,6,7,2024-03-26 00:00:00+00:00,click,"{'device': 'mobile', 'page_url': '/category/7', 'element_id': 'btn_7'}"
6,7,8,2024-03-25 00:00:00+00:00,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
7,8,9,2024-03-24 00:00:00+00:00,checkout,"{'total': 58.0, 'order_id': 'o_100008', 'items_count': 9, 'payment_method': 'crypto'}"
8,9,10,2024-03-23 00:00:00+00:00,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
9,10,11,2024-03-22 00:00:00+00:00,page_view,"{'url': '/product/11', 'device': 'desktop', 'referrer': 'email'}"


**JSONB Query 1. Containment @>**

In [221]:
sql = '''
SELECT event_id, user_id, event_ts, payload
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND payload @> jsonb_build_object('payment_method', 'card')
LIMIT 10;
'''

pd.read_sql(sql, engine)

,event_id,user_id,event_ts,payload
0,3,4,2024-03-29 00:00:00+00:00,"{'total': 53.0, 'order_id': 'o_100003', 'items_count': 4, 'payment_method': 'card'}"
1,18,19,2024-03-14 00:00:00+00:00,"{'total': 68.0, 'order_id': 'o_100018', 'items_count': 9, 'payment_method': 'card'}"
2,33,34,2024-02-28 00:00:00+00:00,"{'total': 83.0, 'order_id': 'o_100033', 'items_count': 4, 'payment_method': 'card'}"
3,48,49,2024-02-13 00:00:00+00:00,"{'total': 98.0, 'order_id': 'o_100048', 'items_count': 9, 'payment_method': 'card'}"
4,63,64,2024-03-29 00:00:00+00:00,"{'total': 113.0, 'order_id': 'o_100063', 'items_count': 4, 'payment_method': 'card'}"
5,78,79,2024-03-14 00:00:00+00:00,"{'total': 128.0, 'order_id': 'o_100078', 'items_count': 9, 'payment_method': 'card'}"
6,93,94,2024-02-28 00:00:00+00:00,"{'total': 143.0, 'order_id': 'o_100093', 'items_count': 4, 'payment_method': 'card'}"
7,108,109,2024-02-13 00:00:00+00:00,"{'total': 158.0, 'order_id': 'o_100108', 'items_count': 9, 'payment_method': 'card'}"
8,123,124,2024-03-29 00:00:00+00:00,"{'total': 173.0, 'order_id': 'o_100123', 'items_count': 4, 'payment_method': 'card'}"
9,138,139,2024-03-14 00:00:00+00:00,"{'total': 188.0, 'order_id': 'o_100138', 'items_count': 9, 'payment_method': 'card'}"


**JSONB Query 2. Key existence ? and ?|**

In [222]:
sql = '''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ? 'tags'
LIMIT 10;
'''

pd.read_sql(sql, engine)

,event_id,event_type,payload
0,2,add_to_cart,"{'tags': ['clearance'], 'price': 12.0, 'quantity': 3, 'product_id': 3}"
1,7,add_to_cart,"{'tags': ['new'], 'price': 17.0, 'quantity': 3, 'product_id': 8}"
2,12,add_to_cart,"{'tags': ['promo'], 'price': 22.0, 'quantity': 3, 'product_id': 13}"
3,17,add_to_cart,"{'tags': ['clearance'], 'price': 27.0, 'quantity': 3, 'product_id': 18}"
4,22,add_to_cart,"{'tags': ['new'], 'price': 32.0, 'quantity': 3, 'product_id': 23}"
5,27,add_to_cart,"{'tags': ['promo'], 'price': 37.0, 'quantity': 3, 'product_id': 28}"
6,32,add_to_cart,"{'tags': ['clearance'], 'price': 42.0, 'quantity': 3, 'product_id': 33}"
7,37,add_to_cart,"{'tags': ['new'], 'price': 47.0, 'quantity': 3, 'product_id': 38}"
8,42,add_to_cart,"{'tags': ['promo'], 'price': 52.0, 'quantity': 3, 'product_id': 43}"
9,47,add_to_cart,"{'tags': ['clearance'], 'price': 57.0, 'quantity': 3, 'product_id': 48}"


In [223]:
sql = '''
SELECT event_id, event_type, payload
FROM t7_hw_events
WHERE payload ?| ARRAY['experiment', 'model_version']
LIMIT 10;
'''

pd.read_sql(sql, engine)

,event_id,event_type,payload
0,4,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [5, 6, 7]}"
1,9,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [10, 11, 12]}"
2,14,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [15, 16, 17]}"
3,19,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [20, 21, 22]}"
4,24,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [25, 26, 27]}"
5,29,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [30, 31, 32]}"
6,34,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [35, 36, 37]}"
7,39,recommendation_shown,"{'experiment': 'exp_1', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [40, 41, 42]}"
8,44,recommendation_shown,"{'experiment': 'exp_3', 'model_name': 'rec_v5', 'model_version': '1.4', 'recommendations': [45, 46, 47]}"
9,49,recommendation_shown,"{'experiment': 'exp_2', 'model_name': 'rec_v5', 'model_version': '1.9', 'recommendations': [50, 51, 52]}"


**JSONB Query 3. JSONPath**

In [224]:
sql = '''
SELECT
    event_id,
    jsonb_path_query(payload, '$.recommendations[*]') AS recommended_pid
FROM t7_hw_events
WHERE event_type = 'recommendation_shown'
LIMIT 15;
'''

pd.read_sql(sql, engine)

,event_id,recommended_pid
0,4,5
1,4,6
2,4,7
3,9,10
4,9,11
5,9,12
6,14,15
7,14,16
8,14,17
9,19,20


In [225]:
sql = '''
SELECT
    event_id,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total
FROM t7_hw_events
WHERE event_type = 'checkout'
  AND jsonb_path_exists(payload, '$.total ? (@ > 300)')
LIMIT 10;
'''

pd.read_sql(sql, engine)

,event_id,order_id,total
0,253,o_100253,303.0
1,258,o_100258,308.0
2,263,o_100263,313.0
3,268,o_100268,318.0
4,273,o_100273,323.0
5,278,o_100278,328.0
6,283,o_100283,333.0
7,288,o_100288,338.0
8,293,o_100293,343.0
9,298,o_100298,348.0


## 4. GIN-index demo + EXPLAIN ANALYZE

**Containment query without GIN-index**

In [226]:
with engine.begin() as conn:
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_payload_gin_pathops'))
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops'))
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_device_btree'))
    conn.execute(text('ANALYZE t7_hw_events'))

explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''')

Aggregate  (cost=1755.76..1755.77 rows=1 width=8) (actual time=22.365..22.366 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=2302 width=0) (actual time=0.006..22.252 rows=3334 loops=1)
        Filter: (payload @> jsonb_build_object('payment_method', 'card'))
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.124 ms
Execution Time: 22.379 ms


**Baseline check: literal vs `jsonb_build_object(...)`.** `jsonb_build_object` is a STABLE function, so PostgreSQL cannot fold it into a constant and re-evaluates it for every row of the Seq Scan. The cell below runs the same containment predicate with a constant jsonb literal, still without any index. The difference between the two Seq Scans is overhead of the query construct, not of the data access, and must be separated from the real effect of GIN.

In [227]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> '{"payment_method": "card"}'::jsonb;
''')

Aggregate  (cost=1630.76..1630.77 rows=1 width=8) (actual time=6.709..6.710 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=2302 width=0) (actual time=0.006..6.605 rows=3334 loops=1)
        Filter: (payload @> '{"payment_method": "card"}'::jsonb)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning Time: 0.069 ms
Execution Time: 6.723 ms


**GIN-index for containment / JSONPath workload**

In [228]:
sql = '''
CREATE INDEX idx_t7_events_payload_gin_pathops
    ON t7_hw_events USING GIN (payload jsonb_path_ops);
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("GIN-index created successfully!")

GIN-index created successfully!


In [229]:
sql = '''
ANALYZE t7_hw_events;
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Analysis completed successfully!")

Analysis completed successfully!


In [230]:
sql = '''
SELECT pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops'))
       AS gin_index_size;
'''

pd.read_sql(sql, engine)
     

,gin_index_size
0,1144 kB


**Same containment query after GIN**

In [231]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('payment_method', 'card');
''')

Aggregate  (cost=1084.13..1084.14 rows=1 width=8) (actual time=2.846..2.847 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=32.45..1076.75 rows=2953 width=0) (actual time=0.285..2.745 rows=3334 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('payment_method', 'card'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..31.72 rows=2953 width=0) (actual time=0.196..0.196 rows=3334 loops=1)
              Index Cond: (payload @> jsonb_build_object('payment_method', 'card'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=14
Planning Time: 0.230 ms
Execution Time: 2.864 ms


In [232]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @> jsonb_build_object('referrer', 'google');
''')

Aggregate  (cost=1061.85..1061.86 rows=1 width=8) (actual time=2.087..2.087 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=23.12..1056.94 rows=1961 width=0) (actual time=0.219..2.012 rows=2500 loops=1)
        Recheck Cond: (payload @> jsonb_build_object('referrer', 'google'))
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..22.63 rows=1961 width=0) (actual time=0.156..0.156 rows=2500 loops=1)
              Index Cond: (payload @> jsonb_build_object('referrer', 'google'))
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=1
Planning Time: 0.077 ms
Execution Time: 2.100 ms


**Observation.** For both containment queries the plan changed from `Seq Scan` to `Bitmap Index Scan` + `Bitmap Heap Scan`. Matches are spread evenly over the table, so the Bitmap Heap Scan still visits every heap block (`Heap Blocks: exact=...` equals the table size in pages). The gain therefore comes from skipping the per-row `@>` evaluation, not from reading fewer pages. Compare with the literal-predicate baseline above for the fair before/after.

**When GIN does not help: path-specific text equality**

In [233]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''')

Aggregate  (cost=1750.62..1750.63 rows=1 width=8) (actual time=5.417..5.418 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1750.00 rows=250 width=0) (actual time=0.011..5.164 rows=8333 loops=1)
        Filter: ((payload ->> 'device'::text) = 'mobile'::text)
        Rows Removed by Filter: 41667
        Buffers: shared hit=1000
Planning Time: 0.046 ms
Execution Time: 5.431 ms


In [234]:
sql = '''
CREATE INDEX idx_t7_events_device_btree
    ON t7_hw_events ((payload ->> 'device'));
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Expression index created successfully!")
     

Expression index created successfully!


In [235]:
sql = '''
ANALYZE t7_hw_events;
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Analysis completed successfully!")

Analysis completed successfully!


In [236]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ->> 'device' = 'mobile';
''')

Aggregate  (cost=1240.60..1240.61 rows=1 width=8) (actual time=1.718..1.718 rows=1 loops=1)
  Buffers: shared hit=1000 read=9
  ->  Bitmap Heap Scan on t7_hw_events  (cost=96.23..1219.98 rows=8250 width=0) (actual time=0.353..1.463 rows=8333 loops=1)
        Recheck Cond: ((payload ->> 'device'::text) = 'mobile'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1000 read=9
        ->  Bitmap Index Scan on idx_t7_events_device_btree  (cost=0.00..94.17 rows=8250 width=0) (actual time=0.264..0.265 rows=8333 loops=1)
              Index Cond: ((payload ->> 'device'::text) = 'mobile'::text)
              Buffers: shared read=9
Planning:
  Buffers: shared hit=5 read=1
Planning Time: 0.100 ms
Execution Time: 1.736 ms


**Observation.** The `->>` query stayed a `Seq Scan` even though the GIN index already existed: a GIN index on `payload` indexes the keys and values of the document, not the text produced by `payload ->> 'device'`. Before the expression index the planner had no statistics for that expression and estimated far fewer rows than the actual count. After the expression b-tree it uses a `Bitmap Index Scan` and the row estimate is close to the actual count.

**Extra: operator classes and the JSONPath operator**

`jsonb_path_ops` (used above) supports only `@>`, `@?` and `@@`. The key-existence operators `?`, `?|`, `?&` need the default `jsonb_ops` class. First, `payload ? 'tags'` with only the `jsonb_path_ops` and b-tree indexes present:

In [237]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
""")

Aggregate  (cost=1650.54..1650.55 rows=1 width=8) (actual time=4.579..4.580 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=10215 width=0) (actual time=0.005..4.271 rows=10000 loops=1)
        Filter: (payload ? 'tags'::text)
        Rows Removed by Filter: 40000
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=7
Planning Time: 0.107 ms
Execution Time: 4.591 ms


In [238]:
with engine.begin() as conn:
    conn.execute(text(
        'CREATE INDEX idx_t7_events_payload_gin_ops '
        'ON t7_hw_events USING GIN (payload jsonb_ops)'
    ))
    conn.execute(text('ANALYZE t7_hw_events'))

pd.read_sql("""
SELECT 'jsonb_path_ops' AS opclass,
       pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')) AS size
UNION ALL
SELECT 'jsonb_ops (default)',
       pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_ops'))
""", engine)

,opclass,size
0,jsonb_path_ops,1144 kB
1,jsonb_ops (default),1696 kB


In [239]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload ? 'tags';
""")

Aggregate  (cost=1231.14..1231.15 rows=1 width=8) (actual time=2.977..2.978 rows=1 loops=1)
  Buffers: shared hit=1006
  ->  Bitmap Heap Scan on t7_hw_events  (cost=78.59..1205.71 rows=10170 width=0) (actual time=0.530..2.675 rows=10000 loops=1)
        Recheck Cond: (payload ? 'tags'::text)
        Heap Blocks: exact=1000
        Buffers: shared hit=1006
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_ops  (cost=0.00..76.05 rows=10170 width=0) (actual time=0.438..0.438 rows=10000 loops=1)
              Index Cond: (payload ? 'tags'::text)
              Buffers: shared hit=6
Planning:
  Buffers: shared hit=18
Planning Time: 0.193 ms
Execution Time: 2.994 ms


**Observation.** With only `jsonb_path_ops` (and the b-tree) available, `payload ? 'tags'` is a `Seq Scan`: that operator class does not support `?`. After creating a default `jsonb_ops` GIN index the same query uses a `Bitmap Index Scan` on `idx_t7_events_payload_gin_ops`. The price is a larger index than `jsonb_path_ops` (sizes above). The demo index is dropped afterwards.

In [240]:
# Remove the demo index again so the following sections run on the same indexes as before
with engine.begin() as conn:
    conn.execute(text('DROP INDEX IF EXISTS idx_t7_events_payload_gin_ops'))
    conn.execute(text('ANALYZE t7_hw_events'))

print("Demo index idx_t7_events_payload_gin_ops dropped")

Demo index idx_t7_events_payload_gin_ops dropped


JSONPath as a *function* versus as an *operator*: `jsonb_path_exists(...)` is an ordinary function call and cannot use a GIN index; the equivalent operator `@?` can (with `jsonb_path_ops`).

In [241]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE jsonb_path_exists(payload, '$.payment_method ? (@ == "card")');
""")

Aggregate  (cost=1666.67..1666.68 rows=1 width=8) (actual time=6.878..6.879 rows=1 loops=1)
  Buffers: shared hit=1000
  ->  Seq Scan on t7_hw_events  (cost=0.00..1625.00 rows=16667 width=0) (actual time=0.009..6.770 rows=3334 loops=1)
        Filter: jsonb_path_exists(payload, '$."payment_method"?(@ == "card")'::jsonpath, '{}'::jsonb, false)
        Rows Removed by Filter: 46666
        Buffers: shared hit=1000
Planning:
  Buffers: shared hit=5
Planning Time: 0.192 ms
Execution Time: 6.893 ms


In [242]:
explain("""
EXPLAIN (ANALYZE, BUFFERS)
SELECT COUNT(*)
FROM t7_hw_events
WHERE payload @? '$.payment_method ? (@ == "card")';
""")

Aggregate  (cost=1090.31..1090.32 rows=1 width=8) (actual time=1.925..1.926 rows=1 loops=1)
  Buffers: shared hit=1004
  ->  Bitmap Heap Scan on t7_hw_events  (cost=35.97..1081.26 rows=3623 width=0) (actual time=0.352..1.821 rows=3334 loops=1)
        Recheck Cond: (payload @? '$."payment_method"?(@ == "card")'::jsonpath)
        Heap Blocks: exact=1000
        Buffers: shared hit=1004
        ->  Bitmap Index Scan on idx_t7_events_payload_gin_pathops  (cost=0.00..35.06 rows=3623 width=0) (actual time=0.228..0.228 rows=3334 loops=1)
              Index Cond: (payload @? '$."payment_method"?(@ == "card")'::jsonpath)
              Buffers: shared hit=4
Planning:
  Buffers: shared hit=9
Planning Time: 0.161 ms
Execution Time: 1.944 ms


**Observation.** The same predicate is a `Seq Scan` when written as the function `jsonb_path_exists(...)` and a `Bitmap Index Scan` on the `jsonb_path_ops` index when written as the operator `@?`. So the GIN index is only used if the query is phrased with an indexable operator.

**Index sizes and write cost**

Read speed-ups are only half of the trade-off. Below: how large each index is compared to the table, and how much slower a bulk `INSERT` of the same 50,000 rows becomes with each index in place.

In [243]:
sql = """
SELECT 'table (heap only)' AS object,
       pg_size_pretty(pg_relation_size('t7_hw_events')) AS size,
       pg_relation_size('t7_hw_events') AS bytes
UNION ALL
SELECT 'primary key (b-tree on event_id)',
       pg_size_pretty(pg_relation_size('t7_hw_events_pkey')),
       pg_relation_size('t7_hw_events_pkey')
UNION ALL
SELECT 'GIN jsonb_path_ops on payload',
       pg_size_pretty(pg_relation_size('idx_t7_events_payload_gin_pathops')),
       pg_relation_size('idx_t7_events_payload_gin_pathops')
UNION ALL
SELECT 'b-tree on (payload ->> ''device'')',
       pg_size_pretty(pg_relation_size('idx_t7_events_device_btree')),
       pg_relation_size('idx_t7_events_device_btree');
"""

sizes = pd.read_sql(sql, engine)
table_bytes = sizes.loc[0, 'bytes']
sizes['pct_of_table'] = (sizes['bytes'] / table_bytes * 100).round(1)
sizes.drop(columns='bytes')

,object,size,pct_of_table
0,table (heap only),8000 kB,100.0
1,primary key (b-tree on event_id),1112 kB,13.9
2,GIN jsonb_path_ops on payload,1144 kB,14.3
3,b-tree on (payload ->> 'device'),360 kB,4.5


In [244]:
import time
from statistics import median

# Identical copy of the events table (no indexes) as a write target.
# Each run: fresh table -> create the indexes under test -> time a bulk INSERT of 50,000 rows.
scenarios = {
    'no index': [],
    'b-tree (device)': [
        "CREATE INDEX ON t7_hw_events_wtest ((payload ->> 'device'))",
    ],
    'GIN jsonb_path_ops': [
        "CREATE INDEX ON t7_hw_events_wtest USING GIN (payload jsonb_path_ops)",
    ],
    'GIN + b-tree': [
        "CREATE INDEX ON t7_hw_events_wtest USING GIN (payload jsonb_path_ops)",
        "CREATE INDEX ON t7_hw_events_wtest ((payload ->> 'device'))",
    ],
}


def time_insert(index_ddls):
    with engine.begin() as conn:
        conn.execute(text('DROP TABLE IF EXISTS t7_hw_events_wtest'))
        conn.execute(text(
            'CREATE TABLE t7_hw_events_wtest '
            '(LIKE t7_hw_events INCLUDING DEFAULTS)'
        ))
        for ddl in index_ddls:
            conn.execute(text(ddl))
        start = time.perf_counter()
        conn.execute(text('''
            INSERT INTO t7_hw_events_wtest (event_id, user_id, event_ts, event_type, payload)
            SELECT event_id, user_id, event_ts, event_type, payload
            FROM t7_hw_events
        '''))
        return (time.perf_counter() - start) * 1000


rows = []
for name, ddls in scenarios.items():
    runs = [time_insert(ddls) for _ in range(5)]
    rows.append({'indexes': name, 'median_ms': round(median(runs), 1)})

with engine.begin() as conn:
    conn.execute(text('DROP TABLE IF EXISTS t7_hw_events_wtest'))

write_cost = pd.DataFrame(rows)
baseline = write_cost.loc[0, 'median_ms']
write_cost['x_vs_no_index'] = (write_cost['median_ms'] / baseline).round(2)
write_cost

,indexes,median_ms,x_vs_no_index
0,no index,43.0,1.00
1,b-tree (device),86.8,2.02
2,GIN jsonb_path_ops,133.8,3.11
3,GIN + b-tree,179.8,4.18


**Caveat.** The test table has no primary key (`LIKE` does not copy it), so "no index" means no index at all. The benchmark is a bulk `INSERT ... SELECT`; row-by-row inserts into a GIN index go through its pending list (`fastupdate`) and behave differently. Absolute timings depend on the machine, the ratios are what matter.

## 5. JSONB vs normalized + Reflection

In [245]:
sql = '''
DROP TABLE IF EXISTS t7_hw_checkouts_norm;

CREATE TABLE t7_hw_checkouts_norm (
    event_id        INTEGER PRIMARY KEY,
    user_id         INTEGER NOT NULL,
    event_ts        TIMESTAMPTZ NOT NULL,
    order_id        TEXT NOT NULL,
    total           NUMERIC(10, 2) NOT NULL,
    items_count     INTEGER NOT NULL CHECK (items_count > 0),
    payment_method  TEXT NOT NULL CHECK (payment_method IN ('card', 'paypal', 'crypto'))
);

INSERT INTO t7_hw_checkouts_norm (
    event_id,
    user_id,
    event_ts,
    order_id,
    total,
    items_count,
    payment_method
)
SELECT
    event_id,
    user_id,
    event_ts,
    payload ->> 'order_id' AS order_id,
    (payload ->> 'total')::NUMERIC(10, 2) AS total,
    (payload ->> 'items_count')::INTEGER AS items_count,
    payload ->> 'payment_method' AS payment_method
FROM t7_hw_events
WHERE event_type = 'checkout';

CREATE INDEX idx_t7_checkouts_norm_pm
    ON t7_hw_checkouts_norm (payment_method);

ANALYZE t7_hw_checkouts_norm;
'''

with engine.begin() as conn:
    conn.execute(text(sql))

print("Normalized table created successfully!")

Normalized table created successfully!


In [246]:
sql = '''
SELECT COUNT(*) AS n_checkouts
FROM t7_hw_checkouts_norm;
'''

pd.read_sql(sql, engine)
     

,n_checkouts
0,10000


**One query, two ways**

**JSONB version**

In [247]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payload ->> 'payment_method' AS payment_method,
    AVG((payload ->> 'total')::NUMERIC) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_events
WHERE event_type = 'checkout'
GROUP BY payload ->> 'payment_method'
ORDER BY avg_total DESC
LIMIT 3;
''')

Limit  (cost=1956.16..1956.17 rows=3 width=72) (actual time=6.860..6.862 rows=3 loops=1)
  Buffers: shared hit=1000
  ->  Sort  (cost=1956.16..1970.08 rows=5569 width=72) (actual time=6.857..6.858 rows=3 loops=1)
        Sort Key: (avg(((payload ->> 'total'::text))::numeric)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=1000
        ->  HashAggregate  (cost=1800.65..1884.18 rows=5569 width=72) (actual time=6.823..6.846 rows=3 loops=1)
              Group Key: (payload ->> 'payment_method'::text)
              Batches: 1  Memory Usage: 217kB
              Buffers: shared hit=1000
              ->  Seq Scan on t7_hw_events  (cost=0.00..1650.09 rows=10037 width=132) (actual time=0.016..4.326 rows=10000 loops=1)
                    Filter: (event_type = 'checkout'::text)
                    Rows Removed by Filter: 40000
                    Buffers: shared hit=1000
Planning:
  Buffers: shared hit=16
Planning Time: 0.149 ms
Execution Time: 6.920 ms


**Normalized version**

In [248]:
explain('''
EXPLAIN (ANALYZE, BUFFERS)
SELECT
    payment_method,
    AVG(total) AS avg_total,
    COUNT(*) AS n
FROM t7_hw_checkouts_norm
GROUP BY payment_method
ORDER BY avg_total DESC
LIMIT 3;
''')

Limit  (cost=269.06..269.07 rows=3 width=46) (actual time=1.918..1.919 rows=3 loops=1)
  Buffers: shared hit=94
  ->  Sort  (cost=269.06..269.07 rows=3 width=46) (actual time=1.917..1.918 rows=3 loops=1)
        Sort Key: (avg(total)) DESC
        Sort Method: quicksort  Memory: 25kB
        Buffers: shared hit=94
        ->  HashAggregate  (cost=269.00..269.04 rows=3 width=46) (actual time=1.910..1.911 rows=3 loops=1)
              Group Key: payment_method
              Batches: 1  Memory Usage: 24kB
              Buffers: shared hit=94
              ->  Seq Scan on t7_hw_checkouts_norm  (cost=0.00..194.00 rows=10000 width=11) (actual time=0.005..0.440 rows=10000 loops=1)
                    Buffers: shared hit=94
Planning:
  Buffers: shared hit=3
Planning Time: 0.102 ms
Execution Time: 1.940 ms


**Observation.** Both plans are `Seq Scan` + `HashAggregate`; the difference is how much data is read. The JSONB query scans the whole `t7_hw_events` table (all 5 event types, the others show up as `Rows Removed by Filter`), extracts and casts values from jsonb row by row, and badly overestimates the number of groups because it has no statistics for `payload ->> 'payment_method'`. The normalized table holds only checkouts in typed columns with statistics, so it reads far fewer buffers. Part of the gap is table size (5x fewer rows, about 10x fewer pages), not only the data model.

**Bonus: MongoDB equivalent (pseudocode, not executed in PostgreSQL)**

```javascript
// MongoDB equivalent: avg total checkout per payment_method, top-3
// Pseudocode - not executed in PostgreSQL

db.events.aggregate([
  {$match: {event_type: "checkout"}},
  {$group: {
    _id: "$payload.payment_method",
    avg_total: {$avg: {$toDouble: "$payload.total"}},
    n: {$sum: 1}
  }},
  {$sort: {avg_total: -1}},
  {$limit: 3}
]);
```

- PostgreSQL JSONB is convenient when you need SQL, JOINs, transactions and a link to normalized tables.
- The MongoDB aggregation pipeline is natural for document-first workloads.
- High-volume raw clickstream may fit better in a document store or a data lake.
- Curated ML features are usually easier to keep in typed SQL tables.

### Reflection

**1. Plans before and after GIN.** The containment query `payload @> jsonb_build_object('payment_method', 'card')` was a Seq Scan (cost 1759.85, 23.8 ms, 46,666 rows removed, 1000 buffers). Part of that is overhead: `jsonb_build_object` is STABLE, so it is re-evaluated per row; with a jsonb literal the Seq Scan costs 1634.85 and takes 6.7 ms. With the `jsonb_path_ops` GIN index the plan became Bitmap Index Scan + Bitmap Heap Scan (cost 1083.97, 2.9 ms, same 3,334 rows): a real gain of about 2.3×, not 8×, and CPU only: all 1000 heap blocks are still read.

**2. GIN vs `->> 'device'`.** GIN indexes document keys and values, so it serves `@>`, but `payload ->> 'device' = 'mobile'` is a text expression it cannot match: the plan stayed a Seq Scan (5.1 ms, estimate 250 rows vs 8,333 actual). A b-tree on that expression gave a Bitmap Index Scan (cost 1242.17, 1.8 ms, estimate 8,312 rows) in only 360 kB.

**3. Read/write trade-off.** Reads speed up; storage and writes pay. GIN takes 1144 kB (14% of the table). Bulk-inserting 50,000 rows took 41.8 ms with no index, 81.3 ms with the b-tree, 113.8 ms with GIN (2.7×) and 156.9 ms with both (3.8×).

**4. JSONB or normalized.** The checkout aggregate took 6.4 ms and 1000 buffers on JSONB versus 1.8 ms and 94 buffers on the typed table (5× fewer rows, 10× fewer pages), and the JSONB plan guessed 5,535 groups instead of 3. I would use JSONB for raw, sparse, fast-changing events and typed tables for curated ML features.

**5. Adding `refund`.** JSONB: insert `refund` events with a new payload; no DDL, optionally a CHECK on required keys. Normalized: create `t7_hw_refunds_norm` with typed columns, constraints and a foreign key to the order, backfill and index.